# German Unemployment Nowcasting with Google Trends

Paper-aligned, reproducible pipeline with modular models (AR, ARX, MIDAS). Run the quickstart cell below to execute the full workflow end-to-end.


In [ ]:
"""
Notebook-local utilities for the nowcasting pipeline.
Keeps everything reproducible without external .py files.
"""
from __future__ import annotations

import dataclasses
import itertools
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, coint


# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
@dataclasses.dataclass
class ModelConfig:
    data_dir: str = "./new_data"
    unemployment_file: str = "./unemployment_rate_germany_DE_only.csv"
    output_dir: str = "./outputs"
    figure_dpi: int = 150

    sample_start: str = "2011-05-01"
    sample_end: str = "2023-01-31"
    eval_start: str = "2019-01-31"
    eval_end: Optional[str] = None

    window_mode: str = "expanding"  # or "rolling"
    rolling_window_months: Optional[int] = None

    target_transform: str = "diff"  # "diff" or "level"
    seasonal_dummies: bool = True

    ar_lag_selection: Dict[str, object] = dataclasses.field(
        default_factory=lambda: {"criterion": "AIC", "max_lags": 6}
    )
    arx_lag_selection: Dict[str, object] = dataclasses.field(
        default_factory=lambda: {
            "criterion": "AIC",
            "max_ar_lags": 4,
            "max_gt_lags": 3,
            "search": "grid",  # or "sequential"
        }
    )

    midas_k_candidates: Sequence[int] = (4, 6, 8, 10, 12)
    midas_ic: str = "AIC"
    include_ar_term: bool = True

    keywords: Sequence[str] = ("indeed", "stepstone", "jobbörse", "arbeitsamt", "bewerbung")
    keyword_file_map: Dict[str, str] = dataclasses.field(
        default_factory=lambda: {
            "indeed": "Indeed",
            "stepstone": "Stepstone",
            "jobbörse": "Jobbörse",
            "arbeitsamt": "Arbeitsamt",
            "bewerbung": "Bewerbung",
        }
    )
    biweekly_schemes: Sequence[str] = ("W12", "W34prev")

    enabled_models: Sequence[str] = ("AR", "ARX", "MIDAS", "MIDAS_restricted")
    include_engle_granger: bool = False


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def set_seed(seed: int = 7) -> None:
    np.random.seed(seed)


def ensure_output_dirs(cfg: ModelConfig) -> None:
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)


def _prep_series(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.tz_localize(None)
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    return df.dropna(subset=["Date", value_col]).set_index("Date").sort_index()


def _median_ratio(a: pd.Series, b: pd.Series) -> float:
    idx = a.index.intersection(b.index)
    x, y = a.loc[idx], b.loc[idx]
    mask = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    if mask.sum() == 0:
        return 1.0
    return np.median((x[mask] / y[mask]).values)


def stitch_three(w1: pd.DataFrame, w2: pd.DataFrame, w3: pd.DataFrame, value_col: str) -> pd.DataFrame:
    W1 = _prep_series(w1, value_col)
    W2 = _prep_series(w2, value_col)
    W3 = _prep_series(w3, value_col)
    s21 = _median_ratio(W1[value_col], W2[value_col])
    W2r = W2 * s21
    s32 = _median_ratio(W2r[value_col], W3[value_col])
    W3r = W3 * s32
    out = pd.concat(
        [W1, W2r.loc[~W2r.index.isin(W1.index)], W3r.loc[~W3r.index.isin(W1.index.union(W2r.index))]]
    ).sort_index()
    return out


def load_gt_weekly(cfg: ModelConfig) -> pd.DataFrame:
    weekly = []
    for kw in cfg.keywords:
        base = cfg.keyword_file_map.get(kw, kw)
        paths = [
            Path(cfg.data_dir) / f"{base}_w1.csv",
            Path(cfg.data_dir) / f"{base}_w2.csv",
            Path(cfg.data_dir) / f"{base}_w3.csv",
        ]
        if not all(p.exists() for p in paths):
            raise FileNotFoundError(f"Missing CSVs for {kw} in {cfg.data_dir}")
        dfs = [pd.read_csv(p, sep=",", skiprows=1) for p in paths]
        stitched = stitch_three(dfs[0], dfs[1], dfs[2], value_col=kw)
        weekly.append(stitched.rename(columns={kw: kw}))
    df_week = weekly[0]
    for s in weekly[1:]:
        df_week = df_week.join(s, how="inner")
    df_week = df_week.loc[(df_week.index >= cfg.sample_start) & (df_week.index <= cfg.sample_end)].sort_index()
    return df_week


def build_biweekly(df_week: pd.DataFrame) -> pd.DataFrame:
    wk = df_week.copy()
    wk.index = wk.index.tz_localize(None)
    wk["day"] = wk.index.day
    w12 = wk[wk["day"] <= 15].groupby(pd.Grouper(freq="M")).mean()
    w34 = wk[wk["day"] > 15].groupby(pd.Grouper(freq="M")).mean().shift(1, freq="M")
    w12.columns = [f"{c}_W12" for c in w12.columns]
    w34.columns = [f"{c}_W34prev" for c in w34.columns]
    return pd.concat([w12, w34], axis=1).sort_index()


def load_unemployment(cfg: ModelConfig) -> pd.Series:
    path = Path(cfg.unemployment_file)
    if not path.exists():
        raise FileNotFoundError(f"Unemployment file missing at {path}")
    data = pd.read_csv(path, sep=",")
    date_col = "Date" if "Date" in data.columns else data.columns[0]
    val_col = [c for c in data.columns if c != date_col][0]
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce").dt.tz_localize(None)
    data = data.dropna(subset=[date_col, val_col]).set_index(date_col).sort_index()
    series = data[val_col]
    return series.loc[(series.index >= cfg.sample_start) & (series.index <= cfg.sample_end)]


def make_monthly_panel(cfg: ModelConfig):
    gt_week = load_gt_weekly(cfg)
    biweekly = build_biweekly(gt_week)
    unemp = load_unemployment(cfg)
    panel = biweekly.join(unemp.to_frame(name="Unemp"), how="inner")
    return panel, unemp, gt_week


def add_month_dummies(index: pd.DatetimeIndex) -> pd.DataFrame:
    dummies = pd.get_dummies(index.month, prefix="m", drop_first=True)
    dummies.index = index
    return dummies


def adf_test(series: pd.Series) -> Dict[str, float]:
    s = series.dropna()
    stat, pval, *_ = adfuller(s)
    return {"stat": stat, "pval": pval}


def engle_granger(y: pd.Series, x: pd.Series) -> Dict[str, float]:
    stat, pval, *_ = coint(y.dropna(), x.dropna())
    return {"stat": stat, "pval": pval}


# ------------------------------------------------------------------
# Lag selection
# ------------------------------------------------------------------

def _ic(res, criterion: str) -> float:
    return res.aic if criterion.upper() == "AIC" else res.bic


def select_ar_lag(y: pd.Series, max_lag: int, criterion: str, dummies: Optional[pd.DataFrame]) -> int:
    best = (np.inf, 1)
    for p in range(1, max_lag + 1):
        df = pd.concat([y] + [y.shift(j) for j in range(1, p + 1)], axis=1)
        df.columns = ["y"] + [f"lag{j}" for j in range(1, p + 1)]
        if dummies is not None:
            df = df.join(dummies)
        df = df.dropna()
        if df.empty:
            continue
        y_reg = df["y"]
        X = sm.add_constant(df.drop(columns=["y"]), has_constant="add")
        res = sm.OLS(y_reg, X).fit()
        ic = _ic(res, criterion)
        if ic < best[0]:
            best = (ic, p)
    return best[1]


def _fit_arx_ic(y: pd.Series, x: pd.Series, p: int, q: int, criterion: str, dummies: Optional[pd.DataFrame]) -> float:
    df = pd.concat([y, x], axis=1).rename(columns={y.name: "y", x.name: "x"})
    for j in range(1, p + 1):
        df[f"y_lag{j}"] = df["y"].shift(j)
    for k in range(0, q + 1):
        df[f"x_lag{k}"] = df["x"].shift(k)
    if dummies is not None:
        df = df.join(dummies)
    df = df.dropna()
    if df.empty:
        return np.inf
    y_reg = df["y"]
    X = sm.add_constant(df.drop(columns=["y"]), has_constant="add")
    res = sm.OLS(y_reg, X).fit()
    return _ic(res, criterion)


def select_arx_lags(y: pd.Series, x: pd.Series, cfg: ModelConfig, dummies: Optional[pd.DataFrame]) -> Tuple[int, int]:
    crit = cfg.arx_lag_selection["criterion"]
    max_p = cfg.arx_lag_selection["max_ar_lags"]
    max_q = cfg.arx_lag_selection["max_gt_lags"]
    search = cfg.arx_lag_selection.get("search", "grid")
    best = (np.inf, 1, 0)
    if search == "sequential":
        p_best = select_ar_lag(y, max_p, crit, dummies)
        for q in range(0, max_q + 1):
            ic = _fit_arx_ic(y, x, p_best, q, crit, dummies)
            if ic < best[0]:
                best = (ic, p_best, q)
    else:
        for p, q in itertools.product(range(1, max_p + 1), range(0, max_q + 1)):
            ic = _fit_arx_ic(y, x, p, q, crit, dummies)
            if ic < best[0]:
                best = (ic, p, q)
    return best[1], best[2]


# ------------------------------------------------------------------
# Model result
# ------------------------------------------------------------------
@dataclasses.dataclass
class ModelResult:
    name: str
    predictions: pd.Series
    actuals: pd.Series
    errors: pd.Series
    metrics: Dict[str, float]
    coeff_history: Optional[Dict[pd.Timestamp, pd.Series]] = None


def compute_metrics(pred: pd.Series, act: pd.Series) -> Dict[str, float]:
    err = pred - act
    return {"RMSE": float(np.sqrt(np.mean(err**2))), "MAE": float(np.mean(np.abs(err)))}


def _eval_dates(index: pd.DatetimeIndex, start_eval: str) -> List[pd.Timestamp]:
    return [d for d in index if d >= pd.to_datetime(start_eval)]


def _window(index: pd.DatetimeIndex, t: pd.Timestamp, cfg: ModelConfig) -> pd.DatetimeIndex:
    if cfg.window_mode == "rolling" and cfg.rolling_window_months:
        start = t - pd.DateOffset(months=cfg.rolling_window_months)
        return index[(index > start) & (index < t)]
    return index[index < t]


def _transform_target(series: pd.Series, cfg: ModelConfig) -> pd.Series:
    return series if cfg.target_transform == "level" else series.diff()


def fit_predict_ar(panel: pd.DataFrame, cfg: ModelConfig) -> ModelResult:
    y_level = panel["Unemp"]
    y = _transform_target(y_level, cfg).rename("y")
    dummies = add_month_dummies(y.index) if cfg.seasonal_dummies else None
    p_sel = select_ar_lag(y, cfg.ar_lag_selection["max_lags"], cfg.ar_lag_selection["criterion"], dummies)

    preds, acts, coeffs = [], [], {}
    for t in _eval_dates(y.index, cfg.eval_start):
        train_idx = _window(y.index, t, cfg)
        y_tr = y.loc[train_idx]
        X = pd.concat([y_tr.shift(j) for j in range(1, p_sel + 1)], axis=1)
        X.columns = [f"lag{j}" for j in range(1, p_sel + 1)]
        if cfg.seasonal_dummies:
            X = X.join(dummies.loc[X.index])
        df = pd.concat([y_tr, X], axis=1).dropna()
        if df.empty:
            continue
        y_reg = df["y"]
        X_reg = sm.add_constant(df.drop(columns=["y"]), has_constant="add")
        res = sm.OLS(y_reg, X_reg).fit()
        coeffs[t] = res.params

        X_t = pd.concat([y.shift(j).loc[[t]] for j in range(1, p_sel + 1)], axis=1)
        X_t.columns = [f"lag{j}" for j in range(1, p_sel + 1)]
        if cfg.seasonal_dummies:
            X_t = X_t.join(dummies.loc[[t]])
        X_t = sm.add_constant(X_t, has_constant="add")
        d_pred = float(res.predict(X_t))
        pred_level = y_level.loc[:t].iloc[-2] + d_pred if cfg.target_transform == "diff" else d_pred
        preds.append(pred_level)
        acts.append(y_level.loc[t])

    pred_s = pd.Series(preds, index=_eval_dates(y.index, cfg.eval_start)[: len(preds)], name="AR_nowcast")
    act_s = pd.Series(acts, index=pred_s.index, name="Unemp")
    err_s = pred_s - act_s
    return ModelResult("AR", pred_s, act_s, err_s, compute_metrics(pred_s, act_s), coeffs)


def fit_predict_arx(panel: pd.DataFrame, gt_col: str, cfg: ModelConfig) -> ModelResult:
    y_level = panel["Unemp"]
    x = panel[gt_col]
    y = _transform_target(y_level, cfg).rename("y")
    x_d = x.diff() if cfg.target_transform == "diff" else x
    dummies = add_month_dummies(y.index) if cfg.seasonal_dummies else None
    p_sel, q_sel = select_arx_lags(y, x_d, cfg, dummies)

    preds, acts, coeffs = [], [], {}
    eval_dates = _eval_dates(y.index, cfg.eval_start)
    for t in eval_dates:
        train_idx = _window(y.index, t, cfg)
        y_tr = y.loc[train_idx]
        x_tr = x_d.loc[train_idx]
        df = pd.concat([y_tr, x_tr], axis=1).rename(columns={x_tr.name: "x"})
        for j in range(1, p_sel + 1):
            df[f"y_lag{j}"] = y_tr.shift(j)
        for k in range(0, q_sel + 1):
            df[f"x_lag{k}"] = x_tr.shift(k)
        if cfg.seasonal_dummies:
            df = df.join(dummies)
        reg_df = df.dropna()
        if reg_df.empty:
            continue
        y_reg = reg_df["y"]
        X_reg = sm.add_constant(reg_df.drop(columns=["y"]), has_constant="add")
        res = sm.OLS(y_reg, X_reg).fit()
        coeffs[t] = res.params

        row = {}
        for j in range(1, p_sel + 1):
            row[f"y_lag{j}"] = y.shift(j).loc[t]
        for k in range(0, q_sel + 1):
            row[f"x_lag{k}"] = x_d.shift(k).loc[t]
        if cfg.seasonal_dummies:
            for c in dummies.columns:
                row[c] = dummies.loc[t, c]
        X_t = sm.add_constant(pd.DataFrame([row]), has_constant="add")
        d_pred = float(res.predict(X_t))
        pred_level = y_level.loc[:t].iloc[-2] + d_pred if cfg.target_transform == "diff" else d_pred
        preds.append(pred_level)
        acts.append(y_level.loc[t])

    pred_s = pd.Series(preds, index=eval_dates[: len(preds)], name=f"ARX_{gt_col}")
    act_s = pd.Series(acts, index=pred_s.index, name="Unemp")
    err_s = pred_s - act_s
    return ModelResult(f"ARX_{gt_col}", pred_s, act_s, err_s, compute_metrics(pred_s, act_s), coeffs)


def _weekly_lag_matrix(gt_week: pd.Series, month_ends: pd.DatetimeIndex, K: int) -> pd.DataFrame:
    rows, idx = [], []
    for m in month_ends:
        weekly_slice = gt_week.loc[gt_week.index <= m].tail(K)
        if len(weekly_slice) < K:
            continue
        rows.append({f"lag{k}": weekly_slice.iloc[-(k + 1)] for k in range(K)})
        idx.append(m)
    return pd.DataFrame(rows, index=pd.to_datetime(idx))


def fit_predict_midas(gt_week: pd.Series, unemp: pd.Series, cfg: ModelConfig, restricted: bool = False) -> ModelResult:
    y_level = unemp
    y = _transform_target(y_level, cfg).rename("y")
    month_ends = y.index
    dummies = add_month_dummies(month_ends) if cfg.seasonal_dummies else None

    best_ic, best_K = np.inf, None
    for K in cfg.midas_k_candidates:
        X_all = _weekly_lag_matrix(gt_week, month_ends, K)
        if cfg.target_transform == "diff":
            X_all = X_all.diff()
        X_all = X_all.reindex(month_ends)
        if restricted:
            X_all = X_all.mean(axis=1).to_frame("avg_week")
        if cfg.include_ar_term:
            X_all["ar1"] = y.shift(1)
        if cfg.seasonal_dummies:
            X_all = X_all.join(dummies)
        df = pd.concat([y, X_all], axis=1).dropna()
        if df.empty:
            continue
        y_reg = df["y"]
        X_reg = sm.add_constant(df.drop(columns=["y"]), has_constant="add")
        res = sm.OLS(y_reg, X_reg).fit()
        ic = _ic(res, cfg.midas_ic)
        if ic < best_ic:
            best_ic, best_K = ic, K

    if best_K is None:
        raise RuntimeError("MIDAS estimation failed")

    preds, acts, coeffs = [], [], {}
    eval_dates = _eval_dates(month_ends, cfg.eval_start)
    for t in eval_dates:
        X_all = _weekly_lag_matrix(gt_week, month_ends[month_ends <= t], best_K)
        if cfg.target_transform == "diff":
            X_all = X_all.diff()
        X_all = X_all.reindex(month_ends[month_ends <= t])
        if restricted:
            X_all = X_all.mean(axis=1).to_frame("avg_week")
        if cfg.include_ar_term:
            X_all["ar1"] = y.shift(1)
        if cfg.seasonal_dummies:
            X_all = X_all.join(dummies)

        train_idx = _window(X_all.index, t, cfg)
        df_train = pd.concat([y.loc[train_idx], X_all.loc[train_idx]], axis=1).dropna()
        if df_train.empty:
            continue
        y_reg = df_train["y"]
        X_reg = sm.add_constant(df_train.drop(columns=["y"]), has_constant="add")
        res = sm.OLS(y_reg, X_reg).fit()
        coeffs[t] = res.params

        X_t = sm.add_constant(X_all.loc[[t]], has_constant="add")
        d_pred = float(res.predict(X_t))
        pred_level = y_level.loc[:t].iloc[-2] + d_pred if cfg.target_transform == "diff" else d_pred
        preds.append(pred_level)
        acts.append(y_level.loc[t])

    name = f"MIDAS_{gt_week.name}" + ("_restricted" if restricted else "")
    pred_s = pd.Series(preds, index=eval_dates[: len(preds)], name=name)
    act_s = pd.Series(acts, index=pred_s.index, name="Unemp")
    err_s = pred_s - act_s
    return ModelResult(name, pred_s, act_s, err_s, compute_metrics(pred_s, act_s), coeffs)


# ------------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------------

def diebold_mariano(e1: pd.Series, e2: pd.Series, h: int = 1, crit: str = "MSE") -> Tuple[float, float]:
    common = e1.index.intersection(e2.index)
    d = (e1.loc[common] ** 2 - e2.loc[common] ** 2) if crit == "MSE" else (np.abs(e1.loc[common]) - np.abs(e2.loc[common]))
    d = d.dropna()
    if d.empty:
        return np.nan, np.nan
    T = len(d)
    d_bar = d.mean()
    gamma = [np.sum((d - d_bar)[:-k] * (d - d_bar)[k:]) / T for k in range(h)]
    var = gamma[0] + 2 * np.sum(gamma[1:])
    dm_stat = d_bar / np.sqrt(var / T)
    pval = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return dm_stat, pval


def results_table(results: List[ModelResult]) -> pd.DataFrame:
    rows = [{"model": r.name, "RMSE": r.metrics["RMSE"], "MAE": r.metrics["MAE"]} for r in results]
    return pd.DataFrame(rows).sort_values("RMSE")


# ------------------------------------------------------------------
# Runner
# ------------------------------------------------------------------

def run_all_models(cfg: Optional[ModelConfig] = None) -> Dict[str, object]:
    cfg = cfg or ModelConfig()
    ensure_output_dirs(cfg)
    set_seed(7)

    panel, unemp, gt_week = make_monthly_panel(cfg)
    results: List[ModelResult] = []

    if "AR" in cfg.enabled_models:
        results.append(fit_predict_ar(panel, cfg))

    if "ARX" in cfg.enabled_models:
        for scheme in cfg.biweekly_schemes:
            cols = [c for c in panel.columns if c.endswith(f"_{scheme}")]
            for col in cols:
                results.append(fit_predict_arx(panel, col, cfg))

    if "MIDAS" in cfg.enabled_models:
        for kw in cfg.keywords:
            series = gt_week[kw]
            results.append(fit_predict_midas(series, unemp, cfg, restricted=False))
            if "MIDAS_restricted" in cfg.enabled_models:
                results.append(fit_predict_midas(series, unemp, cfg, restricted=True))

    leaderboard = results_table(results)
    leaderboard.to_csv(Path(cfg.output_dir) / "leaderboard.csv", index=False)

    ar_res = next((r for r in results if r.name == "AR"), None)
    dm_rows = []
    if ar_res is not None:
        for r in results:
            if r is ar_res:
                continue
            dm_stat, pval = diebold_mariano(r.errors, ar_res.errors)
            dm_rows.append({"model": r.name, "dm_stat": dm_stat, "p_value": pval})
    dm_df = pd.DataFrame(dm_rows)
    dm_df.to_csv(Path(cfg.output_dir) / "dm_vs_ar.csv", index=False)

    return {
        "config": cfg,
        "panel": panel,
        "unemp": unemp,
        "gt_week": gt_week,
        "results": results,
        "leaderboard": leaderboard,
        "dm": dm_df,
    }



In [ ]:
# Quickstart (uses notebook-defined utilities above)
quickstart_config = ModelConfig()
results_bundle = run_all_models(quickstart_config)
results_bundle['leaderboard']


## 0. Setup & config
Central configuration object controls sample window, transformations, lag selection, model toggles, and output paths.


In [ ]:
import pandas as pd
pd.set_option('display.float_format', lambda x: f"{x:,.4f}")

config = ModelConfig(
    sample_start="2011-05-01",
    sample_end="2023-01-31",
    eval_start="2019-01-31",
    eval_end=None,
    target_transform="diff",  # level | diff
    seasonal_dummies=True,
    window_mode="expanding",
    ar_lag_selection={"criterion": "AIC", "max_lags": 6},
    arx_lag_selection={"criterion": "AIC", "max_ar_lags": 4, "max_gt_lags": 3, "search": "grid"},
    midas_k_candidates=(4, 6, 8, 10, 12),
    include_ar_term=True,
    keywords=("indeed", "stepstone", "jobbörse", "arbeitsamt", "bewerbung"),
    biweekly_schemes=("W12", "W34prev"),
    enabled_models=("AR", "ARX", "MIDAS", "MIDAS_restricted"),
)

config


## 1. Data loading
Unemployment (monthly) and Google Trends (weekly w1–w3 per keyword). Weekly series are stitched, aggregated to biweekly W12/W34prev, and merged to a monthly panel aligned to month-end labels.


In [ ]:
panel, unemp, gt_week = make_monthly_panel(config)
print(panel.head())
print("Panel shape:", panel.shape)
print("Weekly GT coverage:", gt_week.index.min(), "to", gt_week.index.max())


## 2. Preprocessing
- Date handling: timezone-naive DatetimeIndex.
- Stitch weekly GT windows, create biweekly W12 and W34prev.
- Target transformation: levels or first differences (default Δ).
- Seasonal dummies: monthly with baseline Jan (toggle).
- Mapping for MIDAS weekly lags: lag0 = last week within month t, lag1 = week t−1, … lagK mirrors K most recent weeks without peeking beyond month t.


In [ ]:
d_unemp = unemp.diff()
print("ADF ΔUnemp:", adf_test(d_unemp))
biweekly_cols = [c for c in panel.columns if c != 'Unemp']
print("Biweekly features:", biweekly_cols[:5], "... total", len(biweekly_cols))


## 3. Diagnostics
Augmented Dickey–Fuller for stationarity; optional Engle–Granger cointegration tests can be enabled via config.


In [ ]:
adf_results = {"Unemp_diff": adf_test(d_unemp)}
if config.include_engle_granger:
    eg = {}
    for kw in config.keywords:
        eg[kw] = engle_granger(unemp, gt_week[kw])
    adf_results["engle_granger"] = eg
adf_results


## 4. Models
- AR benchmark with automatic lag selection (AIC/BIC).
- ARX biweekly (grid over AR and GT lags; include_ar_term toggle).
- MIDAS weekly (candidate K set; unrestricted vs restricted = averaged weekly weight).
Each exposes `fit_predict(train_df, test_t, config)` internally and returns standardized `ModelResult` with metrics and coefficient history.


## 5. Pseudo real-time design
Expanding window (rolling optional), information set respects publication delay: at month *t* unemployment known through *t-1*; weekly GT available through final week of *t*.


## 6. Evaluation & Diebold–Mariano
RMSE/MAE computed on evaluation window; pairwise DM tests vs AR benchmark (squared-error loss).


In [ ]:
results = run_all_models(config)
leaderboard = results["leaderboard"]
dm_vs_ar = results["dm"]
leaderboard


## 7. Results summary
Leaderboard sorted by RMSE, best-by-metric pointers, DM tests vs AR.


In [ ]:
leaderboard.to_csv("./outputs/leaderboard.csv", index=False)
dm_vs_ar.to_csv("./outputs/dm_vs_ar.csv", index=False)

best_rmse = leaderboard.iloc[0]
best_mae = leaderboard.sort_values("MAE").iloc[0]
print("Best by RMSE:", best_rmse.model)
print("Best by MAE:", best_mae.model)
print("DM vs AR:\n", dm_vs_ar)

leaderboard


## 8. Plots
Comparison of unemployment vs nowcasts for top models and absolute error over time. Plots saved to `outputs/`.


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

outdir = Path(config.output_dir)
outdir.mkdir(parents=True, exist_ok=True)

# pick best two models
best_models = leaderboard.head(2).model.tolist()
model_map = {r.name: r for r in results["results"]}

plt.figure(figsize=(10,4))
plt.plot(results["unemp"], label="Unemployment")
for name in best_models:
    res = model_map[name]
    plt.plot(res.predictions, label=name)
plt.title("Unemployment vs nowcasts")
plt.legend(); plt.tight_layout()
plt.savefig(outdir / "nowcasts_compare.png", dpi=150)
plt.close()

plt.figure(figsize=(10,3))
for name in best_models:
    res = model_map[name]
    abs_err = res.errors.abs()
    plt.plot(abs_err, label=name)
plt.title("Absolute error over time")
plt.legend(); plt.tight_layout()
plt.savefig(outdir / "absolute_error.png", dpi=150)
plt.close()

"Saved plots", list(outdir.glob("*.png"))


## 9. Appendix utilities
Helper functions (lag selection, DM test, Engle–Granger) live in `nowcasting_pipeline.py`. Edit the module to adjust behavior; notebook keeps a clean, paper-aligned narrative.
